In this file we will Create a Simple Rag System which is memory augmented RAG which is going to act as the pipeline between the Knowledge Base which the DB (Postgresql) and we have to vectorise those data and store it in the same DB just as a column using PG vector

1) Ingestion
2) Chunking
3) Vectorisation
4) Vector DB

The Imports Needed for this RAG pipeline are below

In [42]:
import os
import re
import getpass
from typing import List
from datetime import datetime

from dotenv import load_dotenv

from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import psycopg2

from pgvector.sqlalchemy import Vector

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import PGVector
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
print("All imports OK")

All imports OK


## LangSmith Setup

In [43]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "linkedin-coach-rag"

print("Tracing :", os.environ["LANGSMITH_TRACING"])
print("Project :", os.environ["LANGSMITH_PROJECT"])
print("Key set :", bool(os.environ["LANGSMITH_API_KEY"]))

Tracing : true
Project : linkedin-coach-rag
Key set : True


## Models

Embeddings use `gemini-embedding-001` (3072 dims). LLM uses `gemini-2.5-flash-lite` via `init_chat_model`.

In [44]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
llm        = init_chat_model("google_genai:gemini-2.5-flash-lite")

print("[OK] Embeddings and LLM ready")

[OK] Embeddings and LLM ready


## Ingestion

Loads all rows from `post_versions` joined with `posts` and returns them as LangChain `Document` objects. First run loads everything into the vector DB. Incremental indexing (`last_indexed_at`) will be layered on top later.

In [45]:
def load_docs_from_db(engine) -> list[Document]:
    with engine.connect() as conn:
        rows = conn.execute(text("""
            SELECT pv.id AS version_id, pv.post_id, pv.version_number,
                   pv.content, pv.created_at, p.title
            FROM post_versions pv
            JOIN posts p ON p.id = pv.post_id
            ORDER BY pv.created_at ASC
        """)).fetchall()

    docs = [
        Document(
            page_content=row.content,
            metadata={
                "post_id":        str(row.post_id),
                "version_id":     str(row.version_id),
                "version_number": row.version_number,
                "title":          row.title,
                "created_at":     row.created_at.isoformat(),
            }
        )
        for row in rows
    ]
    print(f"[OK] {len(docs)} document(s) loaded from DB")
    return docs


engine = create_engine(os.getenv("DATABASE_URL"))
docs   = load_docs_from_db(engine)

[OK] 10 document(s) loaded from DB


## Cleaning

Collapses all whitespace sequences (spaces, tabs, newlines) into a single space before chunking. Operates in-memory only - the original `post_versions.content` in the DB is never touched.

In [46]:
def clean_docs(docs: list[Document]) -> list[Document]:
    for doc in docs:
        doc.page_content = re.sub(r'\s+', ' ', doc.page_content).strip()
    return docs


docs = clean_docs(docs)

## Chunking

`split_documents` reads `page_content` from each Document, splits by the configured size, and carries the metadata (`post_id`, `version_id`, `title` etc.) forward into every chunk automatically.

In [47]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=60)

chunks = splitter.split_documents(docs)

print(f"[OK] Total chunks: {len(chunks)}")
print(f"[OK] Last chunk preview: ...{chunks[-1].page_content[-50:]}")

[OK] Total chunks: 91
[OK] Last chunk preview: ... de motivation appréciée) à recrutement@arkange.io


The below cell is model loading for chat 

In [48]:

os.environ["GOOGLE_API_KEY"] = os.getenv("LANGCHAIN_API_KEY_GEMINI")

model = init_chat_model("google_genai:gemini-2.5-flash-lite")
print(f"[OK] LLM initialized: {model.model}")

[OK] LLM initialized: gemini-2.5-flash-lite


now we will implement the embeddings model same we use google gemini for this also

In [49]:
os.environ["GOOGLE_API_KEY"] = os.getenv("LANGCHAIN_API_KEY_EMBEDDING_GEMINI")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

print(f"[OK] Embedding model initialised: {embeddings.model}")

[OK] Embedding model initialised: models/gemini-embedding-001


In [50]:
# when we call vector_store which has gemini model loaded will call the chunks and embed them  

vector_store = PGVector(
    collection_name="linkedin_coach_posts",
    connection_string=os.getenv("DATABASE_URL"),
    embedding_function=embeddings,
)

print(f"[OK] Vector store initialised: {vector_store.collection_name}")
print(f"[OK] Adding {len(chunks)} chunks...")
vector_store.add_documents(chunks)
print(f"[OK] Done")

C:\Users\Akash\AppData\Local\Temp\ipykernel_12736\67671490.py:3: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use the new operators. 
  vector_store = PGVector(


[OK] Vector store initialised: linkedin_coach_posts
[OK] Adding 91 chunks...
[OK] Done
